# Advanced: Solar Validation Against HALO-(AC)³

In this notebook, we'll validate pyRadtran's solar broadband simulations against real aircraft measurements from the HALO-(AC)³ campaign. This demonstrates how to compare simulated downwelling irradiance against observed values.

## Setup

In [ ]:
import pyradtran
from pyradtran import load_config
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pathlib import Path
import pandas as pd
import logging

# Configure logging for pyradtran
logging.getLogger('pyradtran').setLevel(logging.CRITICAL)

# ── Simulation parameters ─────────────────────────────────────────────────────
cfg = load_config()

# Radiosonde-driven broadband solar simulation
cfg.simulation_defaults.rte_solver           = "disort"
cfg.simulation_defaults.mol_abs_param        = "reptran per_nm"
cfg.simulation_defaults.source               = "solar"
cfg.simulation_defaults.wavelength_nm        = [400, 4500]
cfg.simulation_defaults.integrate_wavelength = True
cfg.simulation_defaults.h2o_source           = "radiosonde"
cfg.simulation_defaults.albedo_value         = 0.3
cfg.simulation_defaults.surface_temperature_k = 253.15
cfg.simulation_defaults.ozone_du             = 300.0
cfg.simulation_defaults.output_altitudes_km  = [0.0]
cfg.simulation_defaults.output_columns       = ["zout", "lambda", "sza", "edir", "eglo", "edn", "eup", "enet", "albedo"]
cfg.execution.max_workers                    = 16
cfg.execution.cleanup_temp_files             = False

config_path = Path("config/radiosonde.yaml")
cfg.to_yaml(config_path)
print(f"Config saved to {config_path}")

## Loading Measurement Data

```{note}
This notebook requires the HALO-(AC)³ broadband radiation CSV file bundled in the `data/` directory. The dataset contains clear-sky aircraft measurements from the HALO, P5, and P6 aircraft.
```

In [ ]:
# Load the CSV file and convert to xarray Dataset
df = pd.read_csv(
    'data/HALO-AC3_HALO_P5_P6_aircraft_broadband_radiation_clear_sky_with_ocean_100s.csv',
    parse_dates=['time'],
)
df = df.set_index('time')
ds = xr.Dataset.from_dataframe(df).interpolate_na('time')
# The albedo has over 30% missing values, so we interpolate for now

## Running Simulations

In [ ]:
# Run a spectral simulation for the dataset
print("Running batch spectral simulation...")
ds_sim = ds.pyradtran.run(
    config_path=config_path,
    return_dataset=True,
    save_to_file=True,
    output_path='data/Simulated_HALO-AC3_HALO_aircraft_broadband_radiation_clear_sky_with_ocean_600s.nc',
    albedo_var='albedo',
)

print("\nSimulation complete!")
ds_sim

## Results

We compare the simulated albedo at different output altitudes against the aircraft measurements. The scatter plot on the right shows the 1:1 correlation — points close to the diagonal line indicate good agreement between simulation and observation.

In [ ]:
albedo_sim_z0 = ds_sim.albedo.isel(altitude=0)
albedo_sim_z10 = ds_sim.albedo.isel(altitude=10)
albedo_meas = ds.albedo

fig, (ax, ax_scatter) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [2, 1]}, figsize=(12, 4))
ax.plot(albedo_sim_z10, label='Simulated Albedo at 10m', linestyle='-', marker='x', color='black', alpha=0.7)
ax.plot(albedo_meas, label='Measured Albedo', linestyle='--', marker='o', color='blue', alpha=0.7)
ax.plot(albedo_sim_z0, label='Simulated Albedo at 0m', linestyle='-', marker='s', color='green', alpha=0.7  )
ax.grid(alpha=0.3)
ax.set_xlabel('Time')
ax.set_ylabel('Albedo')
ax.legend(loc='upper right')

ax_scatter.scatter(albedo_meas, albedo_sim_z0, label='sim zout 0m', color='green', alpha=0.5)
ax_scatter.scatter(albedo_meas, albedo_sim_z10, label='sim zout 10m', color='black', alpha=0.5)
ax_scatter.plot([0, 1], [0, 1], linestyle='--', color='k', alpha=0.5)
ax_scatter.set_xlabel('Measured Albedo')
ax_scatter.set_ylabel('Simulated Albedo')
ax_scatter.legend(loc='upper left')
